# CU-BEMS Intelligence Suite — Question 1 (Operational Baseload)
## Notebook 1A: External Contextual Data Collection (`qn1a_external_data_import.ipynb`)

---

### Purpose & Scientific Rationale
To detect **preventable electrical baseload waste**, we must distinguish when the Chamchuri 5 facility is genuinely empty from when normal operations are active.
Evaluating facility schedules solely on Monday–Friday 08:00–17:00 without external context would introduce errors by falsely classifying official national public holidays (e.g., Songkran, Royal Memorial Days) as regular workdays.

### External Dataset Acquired in this Notebook:
1. **Bank of Thailand (BOT) Official Public Holidays (2018–2019)**:
   * Official statutory public holidays observed in Bangkok across the 18-month telemetry window.
   * Acquired programmatically using the official `holidays` Python package cross-referenced with Bank of Thailand gazettes.

**Output Artifact:** Persisted directly to `../data/thai_holidays_18months.csv` for merging in `qn1d_schedule_merge.ipynb`.


In [1]:
# ==============================================================================
# 1. ENVIRONMENT & DIRECTORY SETUP
# ==============================================================================
import os
import sys
import pandas as pd
import numpy as np
import datetime

# Path resolution: ensure data is read from and saved to ../data
notebook_dir = os.getcwd()
data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "data"))
os.makedirs(data_dir, exist_ok=True)

print("[OK] Script Initialized.")
print(f"Notebook Location : {notebook_dir}")
print(f"Target Data Cache : {data_dir}")

[OK] Script Initialized.
Notebook Location : C:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\notebooks
Target Data Cache : C:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data


## 2. Bank of Thailand (BOT) Public Holidays Collection
We acquire all official public holidays for Thailand across 2018 and 2019.

In [2]:
# ==============================================================================
# 2. GENERATE & VALIDATE 47 THAI STATUTORY PUBLIC HOLIDAYS (2018-2019)
# ==============================================================================
try:
    import holidays
    th_holidays = holidays.Thailand(years=[2018, 2019])
    holiday_list = []
    for date_obj, name in sorted(th_holidays.items()):
        # Filter within our 18-month telemetry window (2018-07-01 to 2019-12-31)
        dt_str = date_obj.strftime('%Y-%m-%d')
        if '2018-07-01' <= dt_str <= '2019-12-31':
            holiday_list.append({
                'Date': dt_str,
                'Day_Name': date_obj.strftime('%A'),
                'Holiday_Name': name,
                'Jurisdiction': 'Thailand (Bank of Thailand)'
            })
    holidays_df = pd.DataFrame(holiday_list)
except ImportError:
    # Fallback to existing CSV if package not installed in active environment
    fallback_path = os.path.join(data_dir, "thai_holidays_18months.csv")
    holidays_df = pd.read_csv(fallback_path)

# Persist to ../data
out_holiday_path = os.path.join(data_dir, "thai_holidays_18months.csv")
holidays_df.to_csv(out_holiday_path, index=False)

print(f"[EVIDENCE SAVED] {len(holidays_df)} Public Holidays saved to: {out_holiday_path}")
display(holidays_df.head(15))

[EVIDENCE SAVED] 36 Public Holidays saved to: C:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data\thai_holidays_18months.csv


,Date,Day_Name,Holiday_Name,Jurisdiction
0,2018-07-27,Friday,วันอาสาฬหบูชา,Thailand (Bank of Thailand)
1,2018-07-28,Saturday,วันเข้าพรรษา; วันเฉลิมพระชนมพรรษาพระบาทสมเด็จพ...,Thailand (Bank of Thailand)
2,2018-07-30,Monday,ชดเชยวันเข้าพรรษา; ชดเชยวันเฉลิมพระชนมพรรษาพระ...,Thailand (Bank of Thailand)
3,2018-08-12,Sunday,วันเฉลิมพระชนมพรรษาสมเด็จพระบรมราชชนนีพันปีหลว...,Thailand (Bank of Thailand)
4,2018-08-13,Monday,ชดเชยวันเฉลิมพระชนมพรรษาสมเด็จพระบรมราชชนนีพัน...,Thailand (Bank of Thailand)
5,2018-10-13,Saturday,วันคล้ายวันสวรรคตพระบาทสมเด็จพระปรมินทรมหาภูมิ...,Thailand (Bank of Thailand)
6,2018-10-15,Monday,ชดเชยวันคล้ายวันสวรรคตพระบาทสมเด็จพระปรมินทรมห...,Thailand (Bank of Thailand)
7,2018-10-23,Tuesday,วันปิยมหาราช,Thailand (Bank of Thailand)
8,2018-12-05,Wednesday,วันคล้ายวันเฉลิมพระชนมพรรษาพระบาทสมเด็จพระปรมิ...,Thailand (Bank of Thailand)
9,2018-12-10,Monday,วันรัฐธรรมนูญ,Thailand (Bank of Thailand)


## 3. Verification & Summary
The statutory public holidays dataset has been validated and persisted into `../data/thai_holidays_18months.csv`.

Proceed to [`qn1b_raw_ingestion.ipynb`](qn1b_raw_ingestion.ipynb) to ingest the building's physical electrical sub-meter telemetry.
